In [3]:
import numpy as np
import pandas as pd

from sklearn.model_selection import (
    train_test_split, StratifiedKFold, KFold,
    cross_validate, GridSearchCV
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.metrics import (
    classification_report, confusion_matrix, f1_score,
    precision_score, recall_score, mean_absolute_error,
    mean_squared_error, r2_score
)

clf_df = pd.read_csv("classification_escalation.csv")
reg_df = pd.read_csv("resolution_time_regression.csv")

display(clf_df.head(), reg_df.head())
print("Classification shape:", clf_df.shape)
print("Regression shape:", reg_df.shape)

,ticket_id,complexity_score,team_experience_years,priority,channel,customer_tenure_months,escalated
0,T0000,2.0,15.0,Medium,Email,61.818896,0
1,T0001,9.0,6.0,Critical,Chat,110.024317,0
2,T0002,10.0,8.0,High,Phone,36.311244,0
3,T0003,2.0,1.0,Medium,Phone,85.127855,0
4,T0004,8.0,14.0,Medium,Chat,18.225059,0


,ticket_id,complexity_score,team_experience_years,priority,channel,customer_tenure_months,resolution_time_hours
0,R0000,7.0,10.0,Medium,Chat,32.638377,8.38724
1,R0001,4.0,15.0,Medium,Chat,45.273760,0.50000
2,R0002,4.0,13.0,Low,Portal,44.878323,0.50000
3,R0003,1.0,10.0,High,Phone,40.095034,0.50000
4,R0004,3.0,15.0,Low,Portal,47.451134,0.50000


Classification shape: (900, 7)
Regression shape: (800, 7)


## 2. Experiment A — Classification

**Problem framing:** Predict whether a support ticket will be escalated (`escalated`). `ticket_id` is an identifier, not a useful predictive feature, so it is excluded.

A stratified 80/20 train/test split preserves the class proportions. The test set is held aside and is not used during model comparison or hyperparameter tuning.

In [4]:
X_clf = clf_df.drop(columns=["ticket_id", "escalated"])
y_clf = clf_df["escalated"]

X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X_clf, y_clf, test_size=0.20, random_state=42, stratify=y_clf
)

print("Training rows:", len(X_train_clf))
print("Test rows:", len(X_test_clf))
print("Full target distribution:")
display(y_clf.value_counts().rename("count").to_frame())
print("Training target proportions:")
display(y_train_clf.value_counts(normalize=True).rename("proportion").to_frame())

Training rows: 720
Test rows: 180
Full target distribution:


,count
escalated,
0,709
1,191


Training target proportions:


,proportion
escalated,
0,0.7875
1,0.2125


 2.1 Preprocessing pipeline

Numeric features: median imputation followed by standardization.  
Categorical features: most-frequent imputation followed by one-hot encoding.

All preprocessing is placed inside a `ColumnTransformer`, which is itself inside each model pipeline. During cross-validation, each fold learns imputation values, scaling parameters, and category encodings only from that fold's training portion. This prevents leakage.

In [6]:
numeric_clf = X_clf.select_dtypes(include=np.number).columns.tolist()
categorical_clf = X_clf.select_dtypes(exclude=np.number).columns.tolist()

preprocessor_clf = ColumnTransformer(
    transformers=[
        ("numeric", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), numeric_clf),
        ("categorical", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), categorical_clf)
    ]
)

print("Numeric columns:", numeric_clf)
print("Categorical columns:", categorical_clf)

Numeric columns: ['complexity_score', 'team_experience_years', 'customer_tenure_months']
Categorical columns: ['priority', 'channel']


2.2 Compare three classifiers with identical 5-fold stratified CV

F1 is the primary metric because the positive class (escalated tickets) is the minority class. The same shuffled `StratifiedKFold` split strategy is used for all three models.

In [7]:
cv_clf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

classifiers = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "k-NN": KNeighborsClassifier(),
    "Decision Tree": DecisionTreeClassifier(random_state=42)
}

classification_results = []
for name, estimator in classifiers.items():
    pipe = Pipeline([
        ("preprocessor", preprocessor_clf),
        ("model", estimator)
    ])
    scores = cross_validate(
        pipe, X_train_clf, y_train_clf,
        cv=cv_clf, scoring="f1", return_train_score=False
    )
    classification_results.append({
        "Model": name,
        "Mean CV F1": scores["test_score"].mean(),
        "Std CV F1": scores["test_score"].std()
    })

classification_results_df = pd.DataFrame(classification_results).sort_values(
    "Mean CV F1", ascending=False
)
display(classification_results_df)

,Model,Mean CV F1,Std CV F1
0,Logistic Regression,0.392651,0.062747
1,k-NN,0.360835,0.050295
2,Decision Tree,0.330666,0.058126


 2.3 Hyperparameter tuning

Logistic Regression has the highest mean CV F1 in the initial comparison, so it is selected for tuning. `GridSearchCV` searches the regularization parameter `C`. The search uses only the training data and the same 5-fold stratified CV.

In [9]:
logistic_pipeline = Pipeline([
    ("preprocessor", preprocessor_clf),
    ("model", LogisticRegression(max_iter=1000, random_state=42))
])

param_grid_clf = {"model__C": [0.01, 0.1, 1, 10, 100]}

grid_clf = GridSearchCV(
    logistic_pipeline,
    param_grid=param_grid_clf,
    cv=cv_clf,
    scoring="f1",
    n_jobs=-1,
    refit=True
)
grid_clf.fit(X_train_clf, y_train_clf)

print("Best parameters:", grid_clf.best_params_)
print("Best mean CV F1:", grid_clf.best_score_)

Best parameters: {'model__C': 10}
Best mean CV F1: 0.4225932078257659


2.4 Final classification test evaluation — run once

The selected, tuned pipeline is now evaluated on the held-out test set. Do not use these test results to keep tuning the model.

In [10]:
best_clf = grid_clf.best_estimator_
y_pred_clf = best_clf.predict(X_test_clf)

print("Test F1:", f1_score(y_test_clf, y_pred_clf))
print("Test precision:", precision_score(y_test_clf, y_pred_clf, zero_division=0))
print("Test recall:", recall_score(y_test_clf, y_pred_clf, zero_division=0))
print("\nConfusion matrix (rows = actual, columns = predicted):")
print(confusion_matrix(y_test_clf, y_pred_clf))
print("\nClassification report:")
print(classification_report(y_test_clf, y_pred_clf, zero_division=0))

Test F1: 0.3137254901960784
Test precision: 0.6153846153846154
Test recall: 0.21052631578947367

Confusion matrix (rows = actual, columns = predicted):
[[137   5]
 [ 30   8]]

Classification report:
              precision    recall  f1-score   support

           0       0.82      0.96      0.89       142
           1       0.62      0.21      0.31        38

    accuracy                           0.81       180
   macro avg       0.72      0.59      0.60       180
weighted avg       0.78      0.81      0.77       180



3. Experiment B — Regression

Problem framing: Predict `resolution_time_hours`. The identifier `ticket_id` is excluded. A separate 80/20 train/test split is created. Unlike classification, regression does not use stratification here.

In [12]:
X_reg = reg_df.drop(columns=["ticket_id", "resolution_time_hours"])
y_reg = reg_df["resolution_time_hours"]

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.20, random_state=42
)

print("Training rows:", len(X_train_reg))
print("Test rows:", len(X_test_reg))
display(y_reg.describe())

Training rows: 640
Test rows: 160


count    800.000000
mean       5.873160
std        4.144974
min        0.500000
25%        2.631319
50%        5.387386
75%        8.345903
max       27.488229
Name: resolution_time_hours, dtype: float64

 3.1 Regression preprocessing

As in classification, numeric columns receive median imputation and scaling, while categorical columns receive most-frequent imputation and one-hot encoding. The preprocessing remains inside the pipeline to prevent leakage across CV folds.

In [13]:
numeric_reg = X_reg.select_dtypes(include=np.number).columns.tolist()
categorical_reg = X_reg.select_dtypes(exclude=np.number).columns.tolist()

preprocessor_reg = ColumnTransformer(
    transformers=[
        ("numeric", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), numeric_reg),
        ("categorical", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), categorical_reg)
    ]
)

print("Numeric columns:", numeric_reg)
print("Categorical columns:", categorical_reg)

Numeric columns: ['complexity_score', 'team_experience_years', 'customer_tenure_months']
Categorical columns: ['priority', 'channel']


 3.2 Compare regression models with 5-fold CV

The metric is MAE. Scikit-learn's scorer returns negative MAE because its model-selection convention maximizes scores, so the code multiplies the fold scores by -1 to report positive MAE values. Lower MAE is better.

In [9]:
cv_reg = KFold(n_splits=5, shuffle=True, random_state=42)

regressors = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(),
    "Decision Tree Regressor": DecisionTreeRegressor(random_state=42)
}

regression_results = []
for name, estimator in regressors.items():
    pipe = Pipeline([
        ("preprocessor", preprocessor_reg),
        ("model", estimator)
    ])
    scores = cross_validate(
        pipe, X_train_reg, y_train_reg,
        cv=cv_reg, scoring="neg_mean_absolute_error",
        return_train_score=False
    )
    fold_mae = -scores["test_score"]
    regression_results.append({
        "Model": name,
        "Mean CV MAE": fold_mae.mean(),
        "Std CV MAE": fold_mae.std()
    })

regression_results_df = pd.DataFrame(regression_results).sort_values(
    "Mean CV MAE", ascending=True
)
display(regression_results_df)

,Model,Mean CV MAE,Std CV MAE
1,Ridge Regression,2.035127,0.171404
0,Linear Regression,2.035134,0.171650
2,Decision Tree Regressor,3.269207,0.143924


3.3 Tune Ridge Regression alpha

Linear Regression and Ridge have almost identical CV MAE. Ridge is selected for tuning because it adds regularization, and its `alpha` parameter is explicitly requested as one of the tuning options in the lab. The search is conducted only on the training data.

In [10]:
ridge_pipeline = Pipeline([
    ("preprocessor", preprocessor_reg),
    ("model", Ridge())
])

param_grid_reg = {"model__alpha": [0.01, 0.1, 1, 10, 100]}

grid_reg = GridSearchCV(
    ridge_pipeline,
    param_grid=param_grid_reg,
    cv=cv_reg,
    scoring="neg_mean_absolute_error",
    n_jobs=-1,
    refit=True
)
grid_reg.fit(X_train_reg, y_train_reg)

print("Best parameters:", grid_reg.best_params_)
print("Best mean CV MAE:", -grid_reg.best_score_)

Best parameters: {'model__alpha': 0.1}
Best mean CV MAE: 2.035118085555478


3.4 Final regression test evaluation — run once

The tuned Ridge pipeline is evaluated once on the held-out test set. MAE is the primary metric; RMSE and R² are included as additional context.

In [11]:
best_reg = grid_reg.best_estimator_
y_pred_reg = best_reg.predict(X_test_reg)

print("Test MAE:", mean_absolute_error(y_test_reg, y_pred_reg))
print("Test RMSE:", mean_squared_error(y_test_reg, y_pred_reg) ** 0.5)
print("Test R²:", r2_score(y_test_reg, y_pred_reg))

Test MAE: 2.2300861441806563
Test RMSE: 3.4027561084563143
Test R²: 0.37746901654220966


4. Required reflections and interpretation

1. Why stratify the classification split?** The target classes are imbalanced. Stratification keeps approximately the same proportion of escalated and non-escalated tickets in both training and test sets, making evaluation more representative.
2. Why must preprocessing be inside the pipeline?** Imputation, scaling, and encoding learn information from data. If they are fitted before CV splitting, validation-fold information can leak into training. A pipeline fits preprocessing only on each fold's training portion.
3. What does fold variability tell you?** The standard deviation of fold scores indicates how much performance changes across different subsets. A larger value suggests less stable performance; a smaller value suggests more consistent performance.
4. Why evaluate the test set only once?** Repeatedly checking the test set and changing the model based on its results indirectly trains decisions on the test data, making the final estimate optimistic. Use CV for selection and reserve the test set for final evaluation.
5. What deployment constraints might change your choice?** Prediction latency, memory/compute cost, model size, interpretability, ease of maintenance, data drift, retraining frequency, and the relative cost of false positives versus false negatives can change the preferred model.

 Outliers: MAE versus RMSE

An outlier is an observation unusually far from the general pattern. In regression, a ticket with an unusually long resolution time can produce a large prediction error. MAE averages absolute errors and is less dominated by a few very large errors. RMSE squares errors before averaging and taking the square root, so large errors receive more influence. Consequently, MAE and RMSE can rank models differently: MAE may favor a model with smaller typical errors, while RMSE may favor a model that avoids a few very large mistakes.

 Final model justification and limitations

The classification model is selected based on the highest mean CV F1 among the three candidates, then tuned with GridSearchCV. Its test confusion matrix should be interpreted in terms of false positives (tickets predicted escalated but not escalated) and false negatives (tickets actually escalated but predicted not escalated). In an operational setting, the cost of missing an escalation may be greater than the cost of reviewing an unnecessary escalation, so recall and the decision threshold may deserve additional attention.

For regression, Ridge is tuned on alpha and compared using CV MAE. Its CV MAE is almost identical to Linear Regression on this dataset, so the practical benefit of regularization is small at the selected setting. The final test MAE is the primary held-out estimate. Results are limited to the available features and sample; unobserved factors, changing ticket patterns, and deployment conditions may affect real-world performance.